In [ ]:
import pandas
from sklearn.preprocessing import PolynomialFeatures


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pandas.read_csv('/content/drive/MyDrive/lending club/current_out_all_clean_data.csv')

In [ ]:
#df['revol_util'].info()
temp_data=df.copy()

# 그리드 서치


In [ ]:
import torch
import torch.nn
import pandas
import numpy
import torch.nn.functional
from torch.autograd import Variable
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from itertools import product
import random


## Prepare Data set
# temp_data = pandas.read_csv('/content/drive/MyDrive/lending club/current_out_all_clean_data.csv')

#temp_data['purpose']

y_temp = temp_data['total_rec_prncp'] / temp_data['funded_amnt']

def clean_data(x):
    x.replace([numpy.inf], 0, inplace=True)
    x.replace([numpy.NAN], 0, inplace=True)

clean_data(y_temp)
# temp_data['revol_util'] = temp_data['revol_util'].str.replace('%', '').astype(float)

# temp_data.drop('id', axis=1, inplace=True)
select_feature = '''
loan_amnt
term
int_rate
installment
sub_grade
emp_length
home_ownership
annual_inc
verification_status
purpose
dti
delinq_2yrs
fico_range_high
mths_since_last_delinq
open_acc
pub_rec
revol_util
total_acc
initial_list_status
last_fico_range_high
collections_12_mths_ex_med
mths_since_last_major_derog
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
max_bal_bc
all_util
total_rev_hi_lim
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_within_12_mths
delinq_amnt
mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
mort_acc
mths_since_recent_bc
mths_since_recent_bc_dlq
mths_since_recent_inq
mths_since_recent_revol_delinq
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats
num_tl_90g_dpd_24m
num_tl_op_past_12m
pct_tl_nvr_dlq
percent_bc_gt_75
pub_rec_bankruptcies
tax_liens
tot_hi_cred_lim
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit
revol_bal_joint
sec_app_fico_range_low
sec_app_fico_range_high
sec_app_inq_last_6mths
sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il
sec_app_num_rev_accts
sec_app_collections_12_mths_ex_med
'''

select_feature = select_feature.strip().split('\n')
x_temp = temp_data[select_feature]

def splitset(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    return x_train.reset_index(drop=True), y_train.reset_index(drop=True), x_valid.reset_index(drop=True), y_valid.reset_index(drop=True), x_test.reset_index(drop=True), y_test.reset_index(drop=True)

bx_train, y_train, bx_valid, y_valid, bx_test, y_test = splitset(x_temp, y_temp)

## Scaling, encoding
def onehot_train_valid(x_train, x_valid):
    one = OneHotEncoder()
    temp_final = pandas.DataFrame()
    temp_valid_final = pandas.DataFrame()
    temp_data = x_train.select_dtypes(include='object')
    temp_data2 = x_valid.select_dtypes(include='object')
    for i in range(0, len(temp_data.columns)):
        one.fit(temp_data.iloc[:, i].values.reshape(-1, 1))
        temp_x = one.transform(temp_data.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x = pandas.DataFrame(temp_x, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])
        temp_x_valid = one.transform(temp_data2.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x_valid = pandas.DataFrame(temp_x_valid, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])

        temp_final = pandas.concat([temp_x, temp_final], axis=1)
        temp_valid_final = pandas.concat([temp_x_valid, temp_valid_final], axis=1)
    return temp_final.reset_index(drop=True), temp_valid_final.reset_index(drop=True)

def stsc_train_valid(x_train, x_valid):
    stdsc = StandardScaler()
    temp_x = x_train.select_dtypes(exclude='object')
    temp_valid = x_valid.select_dtypes(exclude='object')
    name = temp_x.columns
    stdsc.fit(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = stdsc.transform(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = pandas.DataFrame(temp_x)
    temp_x.columns = name
    temp_x_valid = stdsc.transform(numpy.array(temp_valid).reshape(-1, len(name)))
    temp_x_valid = pandas.DataFrame(temp_x_valid)
    temp_x_valid.columns = name

    return temp_x.reset_index(drop=True), temp_x_valid.reset_index(drop=True)

one_train, one_valid = onehot_train_valid(x_train=bx_train, x_valid=bx_valid)
stsc_train, stsc_valid = stsc_train_valid(bx_train, bx_valid)

x_train = pandas.concat([stsc_train, one_train], axis=1)
x_valid = pandas.concat([stsc_valid, one_valid], axis=1)

# Tensor로 전달
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

x_valid_tensor = torch.tensor(x_valid.values, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32)

train_dataset = torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor)
valid_dataset = torch.utils.data.TensorDataset(x_valid_tensor, y_valid_tensor)

# 데이터를 데이터 로더에 전달
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=100)

## 신경망 정의
n_feature = len(x_train.columns)
class LCDNN(torch.nn.Module):
    def __init__(self):
        super(LCDNN, self).__init__()
        self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=256, bias=True)
        self.drop = torch.nn.Dropout(0.25)
        self.fc2 = torch.nn.Linear(in_features=256, out_features=128, bias=True)
        self.fc3 = torch.nn.Linear(in_features=128, out_features=64, bias=True)
        self.bn1 = torch.nn.BatchNorm1d(2**7)
        self.fc4 = torch.nn.Linear(in_features=64, out_features=32, bias=True)
        self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
        self.bn2 = torch.nn.BatchNorm1d(2**6)

    def forward(self, input_data):
        out = input_data.view(-1, n_feature)
        out = torch.nn.functional.relu(self.fc1(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc2(out))
        out = torch.nn.functional.relu(self.bn1(out))
        out = torch.nn.functional.relu(self.fc3(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc4(out))
        out = self.fc5(out)
        return out

# GPU 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 파라미터 정의
learning_rate = 0.001
model = LCDNN()
model.to(device)

criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## optimize
num_epochs = 5
count = 0
loss_list = []
iteration_list = []

predictions_list = []
labels_list = []

# 하이퍼파라미터 범위 설정
learning_rates = [0.001,  0.1]
batch_sizes = [ 128,256]
units_per_layers = [128]
dropout_rates = [0.5]
momentum_values = [0.9]
weight_decay_values = [0]

# 가능한 모든 하이퍼파라미터 조합 생성
param_grid =list(product(learning_rates, batch_sizes, units_per_layers, dropout_rates, momentum_values, weight_decay_values))

best_params = None
best_loss = float('inf')
# 초기화
early_stopping_patience = 10  # 얼리 스탑핑을 위한 인내 횟수

# 그리드 서치 실행
for lr, batch_size, units, dropout, momentum, weight_decay in param_grid:
    print(f"Testing combination: LR={lr}, Batch Size={batch_size}, Units={units}, Dropout={dropout}, Momentum={momentum}, Weight Decay={weight_decay}")

    # 데이터 로더 재설정
    train_loader = DataLoader(train_dataset, batch_size=batch_size)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    # 모델 재정의
    class LCDNN(torch.nn.Module):
        def __init__(self):
            super(LCDNN, self).__init__()
            self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=units, bias=True)
            self.drop = torch.nn.Dropout(dropout)
            self.fc2 = torch.nn.Linear(in_features=units, out_features=units//2, bias=True)
            self.fc3 = torch.nn.Linear(in_features=units//2, out_features=units//4, bias=True)
            self.bn1 = torch.nn.BatchNorm1d(units//2)
            self.fc4 = torch.nn.Linear(in_features=units//4, out_features=32, bias=True)
            self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
            self.bn2 = torch.nn.BatchNorm1d(units//4)

        def forward(self, input_data):
            out = input_data.view(-1, n_feature)
            out = torch.nn.functional.relu(self.fc1(out))
            out = self.drop(out)
            out = torch.nn.functional.relu(self.fc2(out))
            #out = torch.nn.functional.relu(self.bn1(out))
            out = torch.nn.functional.relu(self.fc3(out))
            out = self.drop(out)
            #out = torch.nn.functional.relu(self.fc4(out))
            out = self.fc5(out)
            return out

    # 모델 초기화 및 학습 설정
    model = LCDNN().to(device)
    criterion = torch.nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 얼리 스탑핑 변수 초기화
    best_val_loss = float('inf')
    epochs_no_improve = 0

    # 학습 loop
    count = 0
    for epoch in range(num_epochs):
        model.train()
        for feature, labels in train_loader:
            feature, labels = feature.to(device), labels.to(device)
            train = feature.view(feature.size(0), -1)
            outputs = model(train)
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            count += 1

            if count % 500 == 0:
                model.eval()
                total_loss = 0
                total_rmse = 0.0
                total_mape = 0.0
                total_r2 = 0.0
                num_samples_mape = 0

                with torch.no_grad():
                    for inputs, targets in valid_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        test = inputs.view(inputs.size(0), -1)

                        outputs = model(test)
                        if outputs.size(1) == 1:
                            outputs = outputs.squeeze(1)

                        loss = criterion(outputs, targets)
                        total_loss += loss.item()

                        mse = torch.nn.functional.mse_loss(outputs, targets)
                        rmse = torch.sqrt(mse)
                        total_rmse += rmse.item()

                        epsilon = 1e-8
                        non_zero_targets = torch.abs(targets) > epsilon
                        if torch.sum(non_zero_targets) > 0:
                            mape = torch.mean(torch.abs((targets[non_zero_targets] - outputs[non_zero_targets]) /
                                                         (targets[non_zero_targets] + epsilon))) * 100
                            total_mape += mape.item()
                            num_samples_mape += 1

                        y_mean = torch.mean(targets)
                        ss_tot = torch.sum((targets - y_mean) ** 2)
                        ss_res = torch.sum((targets - outputs) ** 2)

                        # Handle case where ss_tot is zero or very small to avoid NaN
                        if ss_tot.item() > 1e-8:
                            r2 = 1 - (ss_res / ss_tot)
                        else:
                            r2 = 0.0  # Assign 0 if ss_tot is too small
                        total_r2 += r2

                average_loss = total_loss / len(valid_loader)
                average_rmse = total_rmse / len(valid_loader)
                average_mape = total_mape / num_samples_mape if num_samples_mape > 0 else float('nan')
                average_r2 = total_r2 / len(valid_loader) if len(valid_loader) > 0 else float('nan')

                print(f"Iteration: {count}, Validation Loss: {average_loss:.4f}, Average RMSE: {average_rmse:.4f}, "
                      f"Average MAPE: {average_mape:.4f}%, Average R²: {average_r2:.4f}")

                # 얼리 스탑핑 체크
                if average_loss < best_val_loss:
                    best_val_loss = average_loss
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= early_stopping_patience:
                    print("Early stopping triggered")
                    break

        if epochs_no_improve >= early_stopping_patience:
            break

    if best_val_loss < best_loss:
        best_loss = best_val_loss
        best_params = (lr, batch_size, units, dropout, momentum, weight_decay)
        print(f"New best loss: {best_loss} with params: {best_params}")

print(f"Best parameters from grid search: {best_params} with loss: {best_loss}")

Testing combination: LR=0.001, Batch Size=128, Units=128, Dropout=0.5, Momentum=0.9, Weight Decay=0
Iteration: 500, Validation Loss: 0.1564, Average RMSE: 0.2232, Average MAPE: 64.0525%, Average R²: 0.4252
Iteration: 1000, Validation Loss: 0.0995, Average RMSE: 0.2206, Average MAPE: 63.2400%, Average R²: 0.4390
Iteration: 1500, Validation Loss: 0.0938, Average RMSE: 0.2257, Average MAPE: 59.6670%, Average R²: 0.4122
Iteration: 2000, Validation Loss: 0.0922, Average RMSE: 0.2200, Average MAPE: 56.8115%, Average R²: 0.4403
Iteration: 2500, Validation Loss: 0.0930, Average RMSE: 0.2252, Average MAPE: 63.5046%, Average R²: 0.4157
Iteration: 3000, Validation Loss: 0.0918, Average RMSE: 0.2219, Average MAPE: 58.9333%, Average R²: 0.4308
Iteration: 3500, Validation Loss: 0.0920, Average RMSE: 0.2235, Average MAPE: 59.1799%, Average R²: 0.4235
Iteration: 4000, Validation Loss: 0.0913, Average RMSE: 0.2228, Average MAPE: 56.9939%, Average R²: 0.4259
Iteration: 4500, Validation Loss: 0.0922, Ave

In [ ]:
print("re")

In [ ]:
#

## **테스트**
# ```
installment,하영,loan_anmt,total_bc_limit,max_bal_bc
```


In [ ]:
installments = ["installment","loan_amnt","total_bc_limit","max_bal_bc"]

In [ ]:
ilutil = ["il_util","all_util"]

In [ ]:
def poly_feature(data,feature,degree : int) :
    temp_feature=list(feature)
    temp_data=data[temp_feature]
    features = temp_data.columns
    poly = PolynomialFeatures(degree=degree)
    temp_data2=pandas.DataFrame(poly.fit_transform(temp_data),columns=poly.get_feature_names_out(input_features=features))
    temp_data2=temp_data2.drop(['1'],axis=1)
    return pandas.concat([temp_data2,data.drop(temp_feature,axis=1)],axis=1)

In [ ]:
pandas.set_option('display.max_columns',None)

In [ ]:
len(df.columns)

118

In [ ]:
test= poly_feature(test,installments,3)
len(test.columns)

148

In [ ]:
test = df.copy()

In [ ]:
import torch
import torch.nn
import pandas
import numpy
import torch.nn.functional
from torch.autograd import Variable
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from itertools import product
import random


## Prepare Data set
# temp_data = pandas.read_csv('/content/drive/MyDrive/lending club/current_out_all_clean_data.csv')

#temp_data['purpose']

y_temp = temp_data['total_rec_prncp'] / temp_data['funded_amnt']

def clean_data(x):
    x.replace([numpy.inf], 0, inplace=True)
    x.replace([numpy.NAN], 0, inplace=True)

clean_data(y_temp)
# temp_data['revol_util'] = temp_data['revol_util'].str.replace('%', '').astype(float)

# temp_data.drop('id', axis=1, inplace=True)
select_feature = '''
loan_amnt
term
int_rate
installment
sub_grade
emp_length
home_ownership
annual_inc
verification_status
purpose
dti
delinq_2yrs
fico_range_high
mths_since_last_delinq
open_acc
pub_rec
revol_util
total_acc
initial_list_status
last_fico_range_high
collections_12_mths_ex_med
mths_since_last_major_derog
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
max_bal_bc
all_util
total_rev_hi_lim
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_within_12_mths
delinq_amnt
mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
mort_acc
mths_since_recent_bc
mths_since_recent_bc_dlq
mths_since_recent_inq
mths_since_recent_revol_delinq
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats
num_tl_90g_dpd_24m
num_tl_op_past_12m
pct_tl_nvr_dlq
percent_bc_gt_75
pub_rec_bankruptcies
tax_liens
tot_hi_cred_lim
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit
revol_bal_joint
sec_app_fico_range_low
sec_app_fico_range_high
sec_app_inq_last_6mths
sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il
sec_app_num_rev_accts
sec_app_collections_12_mths_ex_med
'''

select_feature = select_feature.strip().split('\n')
x_temp = temp_data[select_feature]

#temp_data= poly_feature(temp_data,installments,3)
temp_data= poly_feature(temp_data,ilutil,3)

def splitset(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    return x_train.reset_index(drop=True), y_train.reset_index(drop=True), x_valid.reset_index(drop=True), y_valid.reset_index(drop=True), x_test.reset_index(drop=True), y_test.reset_index(drop=True)

bx_train, y_train, bx_valid, y_valid, bx_test, y_test = splitset(x_temp, y_temp)

## Scaling, encoding
def onehot_train_valid(x_train, x_valid):
    one = OneHotEncoder()
    temp_final = pandas.DataFrame()
    temp_valid_final = pandas.DataFrame()
    temp_data = x_train.select_dtypes(include='object')
    temp_data2 = x_valid.select_dtypes(include='object')
    for i in range(0, len(temp_data.columns)):
        one.fit(temp_data.iloc[:, i].values.reshape(-1, 1))
        temp_x = one.transform(temp_data.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x = pandas.DataFrame(temp_x, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])
        temp_x_valid = one.transform(temp_data2.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x_valid = pandas.DataFrame(temp_x_valid, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])

        temp_final = pandas.concat([temp_x, temp_final], axis=1)
        temp_valid_final = pandas.concat([temp_x_valid, temp_valid_final], axis=1)
    return temp_final.reset_index(drop=True), temp_valid_final.reset_index(drop=True)

def stsc_train_valid(x_train, x_valid):
    stdsc = StandardScaler()
    temp_x = x_train.select_dtypes(exclude='object')
    temp_valid = x_valid.select_dtypes(exclude='object')
    name = temp_x.columns
    stdsc.fit(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = stdsc.transform(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = pandas.DataFrame(temp_x)
    temp_x.columns = name
    temp_x_valid = stdsc.transform(numpy.array(temp_valid).reshape(-1, len(name)))
    temp_x_valid = pandas.DataFrame(temp_x_valid)
    temp_x_valid.columns = name

    return temp_x.reset_index(drop=True), temp_x_valid.reset_index(drop=True)

one_train, one_valid = onehot_train_valid(x_train=bx_train, x_valid=bx_valid)
stsc_train, stsc_valid = stsc_train_valid(bx_train, bx_valid)

x_train = pandas.concat([stsc_train, one_train], axis=1)
x_valid = pandas.concat([stsc_valid, one_valid], axis=1)

# Tensor로 전달
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

x_valid_tensor = torch.tensor(x_valid.values, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32)

train_dataset = torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor)
valid_dataset = torch.utils.data.TensorDataset(x_valid_tensor, y_valid_tensor)

# 데이터를 데이터 로더에 전달
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=100)

## 신경망 정의
n_feature = len(x_train.columns)
class LCDNN(torch.nn.Module):
    def __init__(self):
        super(LCDNN, self).__init__()
        self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=256, bias=True)
        self.drop = torch.nn.Dropout(0.25)
        self.fc2 = torch.nn.Linear(in_features=256, out_features=128, bias=True)
        self.fc3 = torch.nn.Linear(in_features=128, out_features=64, bias=True)
        self.bn1 = torch.nn.BatchNorm1d(2**7)
        self.fc4 = torch.nn.Linear(in_features=64, out_features=32, bias=True)
        self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
        self.bn2 = torch.nn.BatchNorm1d(2**6)

    def forward(self, input_data):
        out = input_data.view(-1, n_feature)
        out = torch.nn.functional.relu(self.fc1(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc2(out))
        out = torch.nn.functional.relu(self.bn1(out))
        out = torch.nn.functional.relu(self.fc3(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc4(out))
        out = self.fc5(out)
        return out

# GPU 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 파라미터 정의
learning_rate = 0.001
model = LCDNN()
model.to(device)

criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## optimize
num_epochs = 5
count = 0
loss_list = []
iteration_list = []

predictions_list = []
labels_list = []

# 하이퍼파라미터 범위 설정
learning_rates = [0.001,  0.1]
batch_sizes = [ 128,256]
units_per_layers = [128]
dropout_rates = [0.5]
momentum_values = [0.9]
weight_decay_values = [0]

# 가능한 모든 하이퍼파라미터 조합 생성
param_grid =list(product(learning_rates, batch_sizes, units_per_layers, dropout_rates, momentum_values, weight_decay_values))

best_params = None
best_loss = float('inf')
# 초기화
early_stopping_patience = 10  # 얼리 스탑핑을 위한 인내 횟수

# 그리드 서치 실행
for lr, batch_size, units, dropout, momentum, weight_decay in param_grid:
    print(f"Testing combination: LR={lr}, Batch Size={batch_size}, Units={units}, Dropout={dropout}, Momentum={momentum}, Weight Decay={weight_decay}")

    # 데이터 로더 재설정
    train_loader = DataLoader(train_dataset, batch_size=batch_size)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    # 모델 재정의
    class LCDNN(torch.nn.Module):
        def __init__(self):
            super(LCDNN, self).__init__()
            self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=units, bias=True)
            self.drop = torch.nn.Dropout(dropout)
            self.fc2 = torch.nn.Linear(in_features=units, out_features=units//2, bias=True)
            self.fc3 = torch.nn.Linear(in_features=units//2, out_features=units//4, bias=True)
            self.bn1 = torch.nn.BatchNorm1d(units//2)
            self.fc4 = torch.nn.Linear(in_features=units//4, out_features=32, bias=True)
            self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
            self.bn2 = torch.nn.BatchNorm1d(units//4)

        def forward(self, input_data):
            out = input_data.view(-1, n_feature)
            out = torch.nn.functional.relu(self.fc1(out))
            out = self.drop(out)
            out = torch.nn.functional.relu(self.fc2(out))
            #out = torch.nn.functional.relu(self.bn1(out))
            out = torch.nn.functional.relu(self.fc3(out))
            out = self.drop(out)
            #out = torch.nn.functional.relu(self.fc4(out))
            out = self.fc5(out)
            return out

    # 모델 초기화 및 학습 설정
    model = LCDNN().to(device)
    criterion = torch.nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 얼리 스탑핑 변수 초기화
    best_val_loss = float('inf')
    epochs_no_improve = 0

    # 학습 loop
    count = 0
    for epoch in range(num_epochs):
        model.train()
        for feature, labels in train_loader:
            feature, labels = feature.to(device), labels.to(device)
            train = feature.view(feature.size(0), -1)
            outputs = model(train)
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            count += 1

            if count % 500 == 0:
                model.eval()
                total_loss = 0
                total_rmse = 0.0
                total_mape = 0.0
                total_r2 = 0.0
                num_samples_mape = 0

                with torch.no_grad():
                    for inputs, targets in valid_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        test = inputs.view(inputs.size(0), -1)

                        outputs = model(test)
                        if outputs.size(1) == 1:
                            outputs = outputs.squeeze(1)

                        loss = criterion(outputs, targets)
                        total_loss += loss.item()

                        mse = torch.nn.functional.mse_loss(outputs, targets)
                        rmse = torch.sqrt(mse)
                        total_rmse += rmse.item()

                        epsilon = 1e-8
                        non_zero_targets = torch.abs(targets) > epsilon
                        if torch.sum(non_zero_targets) > 0:
                            mape = torch.mean(torch.abs((targets[non_zero_targets] - outputs[non_zero_targets]) /
                                                         (targets[non_zero_targets] + epsilon))) * 100
                            total_mape += mape.item()
                            num_samples_mape += 1

                        y_mean = torch.mean(targets)
                        ss_tot = torch.sum((targets - y_mean) ** 2)
                        ss_res = torch.sum((targets - outputs) ** 2)

                        # Handle case where ss_tot is zero or very small to avoid NaN
                        if ss_tot.item() > 1e-8:
                            r2 = 1 - (ss_res / ss_tot)
                        else:
                            r2 = 0.0  # Assign 0 if ss_tot is too small
                        total_r2 += r2

                average_loss = total_loss / len(valid_loader)
                average_rmse = total_rmse / len(valid_loader)
                average_mape = total_mape / num_samples_mape if num_samples_mape > 0 else float('nan')
                average_r2 = total_r2 / len(valid_loader) if len(valid_loader) > 0 else float('nan')

                print(f"Iteration: {count}, Validation Loss: {average_loss:.4f}, Average RMSE: {average_rmse:.4f}, "
                      f"Average MAPE: {average_mape:.4f}%, Average R²: {average_r2:.4f}")

                # 얼리 스탑핑 체크
                if average_loss < best_val_loss:
                    best_val_loss = average_loss
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= early_stopping_patience:
                    print("Early stopping triggered")
                    break

        if epochs_no_improve >= early_stopping_patience:
            break

    if best_val_loss < best_loss:
        best_loss = best_val_loss
        best_params = (lr, batch_size, units, dropout, momentum, weight_decay)
        print(f"New best loss: {best_loss} with params: {best_params}")

print(f"Best parameters from grid search: {best_params} with loss: {best_loss}")

Testing combination: LR=0.001, Batch Size=128, Units=128, Dropout=0.5, Momentum=0.9, Weight Decay=0
Iteration: 500, Validation Loss: 0.1586, Average RMSE: 0.2239, Average MAPE: 68.1674%, Average R²: 0.4234
Iteration: 1000, Validation Loss: 0.0980, Average RMSE: 0.2204, Average MAPE: 62.0018%, Average R²: 0.4399
Iteration: 1500, Validation Loss: 0.0953, Average RMSE: 0.2227, Average MAPE: 60.8864%, Average R²: 0.4278
Iteration: 2000, Validation Loss: 0.0957, Average RMSE: 0.2257, Average MAPE: 62.6448%, Average R²: 0.4132
Iteration: 2500, Validation Loss: 0.0930, Average RMSE: 0.2257, Average MAPE: 62.5130%, Average R²: 0.4129
Iteration: 3000, Validation Loss: 0.0938, Average RMSE: 0.2218, Average MAPE: 58.6122%, Average R²: 0.4316
Iteration: 3500, Validation Loss: 0.0928, Average RMSE: 0.2263, Average MAPE: 59.4346%, Average R²: 0.4090
Iteration: 4000, Validation Loss: 0.0911, Average RMSE: 0.2207, Average MAPE: 56.9764%, Average R²: 0.4367
Iteration: 4500, Validation Loss: 0.0913, Ave

In [ ]:
import torch
import torch.nn
import pandas
import numpy
import torch.nn.functional
from torch.autograd import Variable
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from itertools import product
import random


## Prepare Data set
# temp_data = pandas.read_csv('/content/drive/MyDrive/lending club/current_out_all_clean_data.csv')

#temp_data['purpose']

y_temp = temp_data['total_rec_prncp'] / temp_data['funded_amnt']

def clean_data(x):
    x.replace([numpy.inf], 0, inplace=True)
    x.replace([numpy.NAN], 0, inplace=True)

clean_data(y_temp)
# temp_data['revol_util'] = temp_data['revol_util'].str.replace('%', '').astype(float)

# temp_data.drop('id', axis=1, inplace=True)
select_feature = '''
loan_amnt
term
int_rate
installment
sub_grade
emp_length
home_ownership
annual_inc
verification_status
purpose
dti
delinq_2yrs
fico_range_high
mths_since_last_delinq
open_acc
pub_rec
revol_util
total_acc
initial_list_status
last_fico_range_high
collections_12_mths_ex_med
mths_since_last_major_derog
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
max_bal_bc
all_util
total_rev_hi_lim
total_cu_tl
inq_last_12m
acc_open_past_24mths
avg_cur_bal
bc_open_to_buy
bc_util
chargeoff_within_12_mths
delinq_amnt
mo_sin_old_il_acct
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
mort_acc
mths_since_recent_bc
mths_since_recent_bc_dlq
mths_since_recent_inq
mths_since_recent_revol_delinq
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_sats
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_sats
num_tl_90g_dpd_24m
num_tl_op_past_12m
pct_tl_nvr_dlq
percent_bc_gt_75
pub_rec_bankruptcies
tax_liens
tot_hi_cred_lim
total_bal_ex_mort
total_bc_limit
total_il_high_credit_limit
revol_bal_joint
sec_app_fico_range_low
sec_app_fico_range_high
sec_app_inq_last_6mths
sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il
sec_app_num_rev_accts
sec_app_collections_12_mths_ex_med
'''

select_feature = select_feature.strip().split('\n')
x_temp = temp_data[select_feature]

#temp_data= poly_feature(temp_data,installments,3)
temp_data= poly_feature(temp_data,ilutil,3)

def splitset(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    return x_train.reset_index(drop=True), y_train.reset_index(drop=True), x_valid.reset_index(drop=True), y_valid.reset_index(drop=True), x_test.reset_index(drop=True), y_test.reset_index(drop=True)

bx_train, y_train, bx_valid, y_valid, bx_test, y_test = splitset(x_temp, y_temp)

## Scaling, encoding
def onehot_train_valid(x_train, x_valid):
    one = OneHotEncoder()
    temp_final = pandas.DataFrame()
    temp_valid_final = pandas.DataFrame()
    temp_data = x_train.select_dtypes(include='object')
    temp_data2 = x_valid.select_dtypes(include='object')
    for i in range(0, len(temp_data.columns)):
        one.fit(temp_data.iloc[:, i].values.reshape(-1, 1))
        temp_x = one.transform(temp_data.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x = pandas.DataFrame(temp_x, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])
        temp_x_valid = one.transform(temp_data2.iloc[:, i].values.reshape(-1, 1)).toarray().astype(int)
        temp_x_valid = pandas.DataFrame(temp_x_valid, columns=[str(one.categories_[0][i]) for i in range(len(one.categories_[0]))])

        temp_final = pandas.concat([temp_x, temp_final], axis=1)
        temp_valid_final = pandas.concat([temp_x_valid, temp_valid_final], axis=1)
    return temp_final.reset_index(drop=True), temp_valid_final.reset_index(drop=True)

def stsc_train_valid(x_train, x_valid):
    stdsc = StandardScaler()
    temp_x = x_train.select_dtypes(exclude='object')
    temp_valid = x_valid.select_dtypes(exclude='object')
    name = temp_x.columns
    stdsc.fit(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = stdsc.transform(numpy.array(temp_x).reshape(-1, len(name)))
    temp_x = pandas.DataFrame(temp_x)
    temp_x.columns = name
    temp_x_valid = stdsc.transform(numpy.array(temp_valid).reshape(-1, len(name)))
    temp_x_valid = pandas.DataFrame(temp_x_valid)
    temp_x_valid.columns = name

    return temp_x.reset_index(drop=True), temp_x_valid.reset_index(drop=True)

one_train, one_valid = onehot_train_valid(x_train=bx_train, x_valid=bx_valid)
stsc_train, stsc_valid = stsc_train_valid(bx_train, bx_valid)

x_train = pandas.concat([stsc_train, one_train], axis=1)
x_valid = pandas.concat([stsc_valid, one_valid], axis=1)

# Tensor로 전달
x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

x_valid_tensor = torch.tensor(x_valid.values, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32)

train_dataset = torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor)
valid_dataset = torch.utils.data.TensorDataset(x_valid_tensor, y_valid_tensor)

# 데이터를 데이터 로더에 전달
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=100)

## 신경망 정의
n_feature = len(x_train.columns)
class LCDNN(torch.nn.Module):
    def __init__(self):
        super(LCDNN, self).__init__()
        self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=256, bias=True)
        self.drop = torch.nn.Dropout(0.25)
        self.fc2 = torch.nn.Linear(in_features=256, out_features=128, bias=True)
        self.fc3 = torch.nn.Linear(in_features=128, out_features=64, bias=True)
        self.bn1 = torch.nn.BatchNorm1d(2**7)
        self.fc4 = torch.nn.Linear(in_features=64, out_features=32, bias=True)
        self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
        self.bn2 = torch.nn.BatchNorm1d(2**6)

    def forward(self, input_data):
        out = input_data.view(-1, n_feature)
        out = torch.nn.functional.relu(self.fc1(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc2(out))
        out = torch.nn.functional.relu(self.bn1(out))
        out = torch.nn.functional.relu(self.fc3(out))
        out = self.drop(out)
        out = torch.nn.functional.relu(self.fc4(out))
        out = self.fc5(out)
        return out

# GPU 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 파라미터 정의
learning_rate = 0.001
model = LCDNN()
model.to(device)

criterion = torch.nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## optimize
num_epochs = 5
count = 0
loss_list = []
iteration_list = []

predictions_list = []
labels_list = []

# 하이퍼파라미터 범위 설정
learning_rates = [0.001,  0.1]
batch_sizes = [ 128,256]
units_per_layers = [128]
dropout_rates = [0.5]
momentum_values = [0.9]
weight_decay_values = [0]

# 가능한 모든 하이퍼파라미터 조합 생성
param_grid =list(product(learning_rates, batch_sizes, units_per_layers, dropout_rates, momentum_values, weight_decay_values))

best_params = None
best_loss = float('inf')
# 초기화
early_stopping_patience = 10  # 얼리 스탑핑을 위한 인내 횟수

# 그리드 서치 실행
for lr, batch_size, units, dropout, momentum, weight_decay in param_grid:
    print(f"Testing combination: LR={lr}, Batch Size={batch_size}, Units={units}, Dropout={dropout}, Momentum={momentum}, Weight Decay={weight_decay}")

    # 데이터 로더 재설정
    train_loader = DataLoader(train_dataset, batch_size=batch_size)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

    # 모델 재정의
    class LCDNN(torch.nn.Module):
        def __init__(self):
            super(LCDNN, self).__init__()
            self.fc1 = torch.nn.Linear(in_features=n_feature, out_features=units, bias=True)
            self.drop = torch.nn.Dropout(dropout)
            self.fc2 = torch.nn.Linear(in_features=units, out_features=units//2, bias=True)
            self.fc3 = torch.nn.Linear(in_features=units//2, out_features=units//4, bias=True)
            self.bn1 = torch.nn.BatchNorm1d(units//2)
            self.fc4 = torch.nn.Linear(in_features=units//4, out_features=32, bias=True)
            self.fc5 = torch.nn.Linear(in_features=32, out_features=1, bias=True)
            self.bn2 = torch.nn.BatchNorm1d(units//4)

        def forward(self, input_data):
            out = input_data.view(-1, n_feature)
            out = torch.nn.functional.relu(self.fc1(out))
            out = self.drop(out)
            out = torch.nn.functional.relu(self.fc2(out))
            #out = torch.nn.functional.relu(self.bn1(out))
            out = torch.nn.functional.relu(self.fc3(out))
            out = self.drop(out)
            #out = torch.nn.functional.relu(self.fc4(out))
            out = self.fc5(out)
            return out

    # 모델 초기화 및 학습 설정
    model = LCDNN().to(device)
    criterion = torch.nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 얼리 스탑핑 변수 초기화
    best_val_loss = float('inf')
    epochs_no_improve = 0

    # 학습 loop
    count = 0
    for epoch in range(num_epochs):
        model.train()
        for feature, labels in train_loader:
            feature, labels = feature.to(device), labels.to(device)
            train = feature.view(feature.size(0), -1)
            outputs = model(train)
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            count += 1

            if count % 500 == 0:
                model.eval()
                total_loss = 0
                total_rmse = 0.0
                total_mape = 0.0
                total_r2 = 0.0
                num_samples_mape = 0

                with torch.no_grad():
                    for inputs, targets in valid_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        test = inputs.view(inputs.size(0), -1)

                        outputs = model(test)
                        if outputs.size(1) == 1:
                            outputs = outputs.squeeze(1)

                        loss = criterion(outputs, targets)
                        total_loss += loss.item()

                        mse = torch.nn.functional.mse_loss(outputs, targets)
                        rmse = torch.sqrt(mse)
                        total_rmse += rmse.item()

                        epsilon = 1e-8
                        non_zero_targets = torch.abs(targets) > epsilon
                        if torch.sum(non_zero_targets) > 0:
                            mape = torch.mean(torch.abs((targets[non_zero_targets] - outputs[non_zero_targets]) /
                                                         (targets[non_zero_targets] + epsilon))) * 100
                            total_mape += mape.item()
                            num_samples_mape += 1

                        y_mean = torch.mean(targets)
                        ss_tot = torch.sum((targets - y_mean) ** 2)
                        ss_res = torch.sum((targets - outputs) ** 2)

                        # Handle case where ss_tot is zero or very small to avoid NaN
                        if ss_tot.item() > 1e-8:
                            r2 = 1 - (ss_res / ss_tot)
                        else:
                            r2 = 0.0  # Assign 0 if ss_tot is too small
                        total_r2 += r2

                average_loss = total_loss / len(valid_loader)
                average_rmse = total_rmse / len(valid_loader)
                average_mape = total_mape / num_samples_mape if num_samples_mape > 0 else float('nan')
                average_r2 = total_r2 / len(valid_loader) if len(valid_loader) > 0 else float('nan')

                print(f"Iteration: {count}, Validation Loss: {average_loss:.4f}, Average RMSE: {average_rmse:.4f}, "
                      f"Average MAPE: {average_mape:.4f}%, Average R²: {average_r2:.4f}")

                # 얼리 스탑핑 체크
                if average_loss < best_val_loss:
                    best_val_loss = average_loss
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= early_stopping_patience:
                    print("Early stopping triggered")
                    break

        if epochs_no_improve >= early_stopping_patience:
            break

    if best_val_loss < best_loss:
        best_loss = best_val_loss
        best_params = (lr, batch_size, units, dropout, momentum, weight_decay)
        print(f"New best loss: {best_loss} with params: {best_params}")

print(f"Best parameters from grid search: {best_params} with loss: {best_loss}")

Testing combination: LR=0.001, Batch Size=128, Units=128, Dropout=0.5, Momentum=0.9, Weight Decay=0
Iteration: 500, Validation Loss: 0.2074, Average RMSE: 0.2454, Average MAPE: 67.4456%, Average R²: 0.3042
Iteration: 1000, Validation Loss: 0.1012, Average RMSE: 0.2246, Average MAPE: 64.8620%, Average R²: 0.4194
Iteration: 1500, Validation Loss: 0.0960, Average RMSE: 0.2240, Average MAPE: 60.5941%, Average R²: 0.4209
Iteration: 2000, Validation Loss: 0.0940, Average RMSE: 0.2201, Average MAPE: 60.6966%, Average R²: 0.4411
Iteration: 2500, Validation Loss: 0.0928, Average RMSE: 0.2212, Average MAPE: 60.8008%, Average R²: 0.4359
Iteration: 3000, Validation Loss: 0.0927, Average RMSE: 0.2219, Average MAPE: 57.1447%, Average R²: 0.4310
Iteration: 3500, Validation Loss: 0.0939, Average RMSE: 0.2231, Average MAPE: 59.9815%, Average R²: 0.4259
Iteration: 4000, Validation Loss: 0.0930, Average RMSE: 0.2196, Average MAPE: 55.4362%, Average R²: 0.4415
Iteration: 4500, Validation Loss: 0.0921, Ave